# ⚙️ Data Preprocessing

---

## 🗂️ Data Structure

The folder structure for the data is organized as follows:

```plaintext
📂 Data storage Siemens
├── 📂 MRI data
│   ├── 📂 single_MRI_machine_1
│   │   ├── 📂 2024-12-11
│   │   │   ├── 📄 Scanner.csv
│   │   │   ├── 📄 Examinations.csv
│   │   │   ├── 📄 Measurements.csv
│   │   │   ├── 📄 Events.csv
│   │   │   ├── 📄 ProtocolParameters.csv
│   │   │   ├── 📄 ProtocolParametersDescription.csv
│   │   │   ├── 📄 powerdata_1.csv
│   │   │   └── ...  
│   │   ├── 📂 2024-12-12
│   │   │   ├── 📄 Scanner.csv
│   │   │   ├── 📄 Examinations.csv
│   │   │   ├── 📄 Measurements.csv
│   │   │   ├── 📄 Events.csv
│   │   │   ├── 📄 ProtocolParameters.csv
│   │   │   ├── 📄 ProtocolParametersDescription.csv
│   │   │   ├── 📄 powerdata_1.csv
│   │   │   └── ...  
│   ├── 📂 single_MRI_machine_2
│   │   ├── 📂 2024-12-11
│   │   │   ├── 📄 Scanner.csv
│   │   │   ├── 📄 Examinations.csv
│   │   │   ├── 📄 Measurements.csv
│   │   │   ├── 📄 Events.csv
│   │   │   ├── 📄 ProtocolParameters.csv
│   │   │   ├── 📄 ProtocolParametersDescription.csv
│   │   │   ├── 📄 powerdata_2.csv
│   │   │   └── ...  
│   │   └── ...
│   └── ...
├── 📂 CT data
├── 📂 Playground
├── 📂 Raw data
...
```

## 📁 Explanation of Folder Structure

### **Raw Data** 🗄️  
- Contains all the data downloaded by running the *Half Automated Data Pulling Pipeline*.  
- Includes both MRI and CT data. The corresponding data is manually copied into <br>
the `MRI data` and `CT data` folders.  

### **MRI Data** 🧲  
- Contains subfolders for each MRI machine (e.g., `single_MRI_machine_1`, `single_MRI_machine_2`).  
- Each machine folder contains subfolders for each date (e.g., `2024-12-11`, `2024-12-12`).  
- Each date folder contains the corresponding data and CSV files.  

### **CT Data** 🩻  
- Similar structure to MRI data, but for CT machines.  

### **Test data** 🛠️  
- Contains a subsample of data used for debugging and testing the data processing pipeline.  

---

## 📄 Explanation of CSV Files

- **`Scanner.csv`**: General information about the scanner and MRI machines.  
- **`Examinations.csv`**: Details about the examinations performed on the MRI machines.  
- **`Measurements.csv`**: Information about individual measurements during the examinations.  
- **`Events.csv`**: Logs of all events, mapping examinations and measurements.  
- **`ProtocolParameters.csv`**: Parameters used in each measurement (abbreviated format).  
- **`ProtocolParametersDescription.csv`**: Explanation of parameter abbreviations used in `ProtocolParameters.csv`.  
- **`powerdata_*`**: Data collected from the powermeters at a 1 Hz sampling rate.  
  - Each scanner is connected to a specific powermeter, and the filename reflects this association.  
  - The filename always starts with `powerdata_`, followed by the name of the <br>
  corresponding powermeter (e.g., `powerdata_Powermeter1.csv`, `powerdata_Powermeter2.csv`).  
  - These files contain detailed power and energy consumption data for the scanner.
---

In [ ]:
# Enable autoreload and black formatting in Jupyter notebooks
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import duckdb as dd
import sys
import glob
import plotly.graph_objects as go
import warnings

# Get the current working directory
CWD = os.getcwd()

GLOBAL_UTILS_DIR = os.path.normpath(
    os.path.join(CWD, "..", "..", "Global utils_opt")
)

# Append this directory to the system path
sys.path.append(GLOBAL_UTILS_DIR)

from global_utils import *

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 🧲 Inferring MRI Scanner Modes from Power Data

---

## 🚦 Scanner Modes

MRI scanners operate in several distinct modes:
- 🌱 **Eco-Power Mode**: Energy-saving mode, typically active during the night.
- 😴 **Idle Mode**: The scanner is powered but not actively scanning, often right after a scan.
- 🧲 **Scanning Mode**: The scanner is actively acquiring images.

---

## ❓ Challenge

- The raw data **does not explicitly indicate** the operational mode of the MRI scanners.
- To **infer the current mode**, we must analyze the **Total Active Power (kW)** over time.

---

## 📈 Approach

1. **Analyze the Total Active Power (kW)**  
   - Examine the power consumption time series for each scanner over a month.
   - Identify characteristic power patterns for each mode.

2. **Mode Inference**  
   - 🌙 **Eco-Power Mode**: Expected during nighttime hours (lowest power usage).
   - ⏱️ **Idle Mode**: Typically follows a scanning session (intermediate power usage).
   - 🧲 **Scanning Mode**: Detected by high power consumption during active scans.

3. **Threshold Determination**  
   - Establish power thresholds that reliably distinguish between the different modes.
   - These thresholds are essential for downstream data processing and analysis.

---

## ⚠️ Important

> **This step must be completed _before_ running the main data preprocessing pipeline!**  
> The preprocessing workflow requires a `powerboundries_scanner_mapping.csv` file  
> in the data directory, containing the determined power thresholds for each scanner.


In [ ]:
def get_data_dir(data_dir: str) -> str:
    """
    Create the path to the data directory and check if it exists.
    If it does, return the path. If it does not, raise an error.

    """
    # Check if the data storage directory exists
    if not os.path.exists(data_dir):
        raise FileNotFoundError(
            f"❌ The data storage directory {data_dir} does not exist."
        )
    else:
        print(f"✅ Data storage directory found: {data_dir}")

    return data_dir


# Construct the path to the data directory
DATA_DIR = os.path.normpath(
    os.path.join(
        CWD, "..", "..", "Data storage Siemens", "Pipeline-V2", "MRI data"
    )
)

# Get the data directory and check if it exists
DATA_DIR = get_data_dir(DATA_DIR)

FileNotFoundError: ❌ The data storage directory c:\Users\flori\Documents\PhD\MRI_data_analysis\Data storage Siemens\Pipeline-V2\MRI data does not exist.

In [ ]:
def set_up_duckdb(cwd: str) -> dd.DuckDBPyConnection:
    """
    Create a DuckDB folder in the current working directory if it does not exist.
    Set up a DuckDB database file in that folder and return a connection to it.

    Parameters:
        cwd (str): The current working directory.

    Returns:
        dd.DuckDBPyConnection: A connection object to the DuckDB database.
    """

    # Create a duckdb folder if it doesn't exist
    duckdb_dir = os.path.join(cwd, "duckdb_files")
    os.makedirs(duckdb_dir, exist_ok=True)
    print(f"✅ DuckDB files directory is at: {duckdb_dir}")

    # Connect to a DuckDB database file in the duckdb_files directory
    duckdb_path = os.path.join(duckdb_dir, "MRI_data.duckdb")
    con = dd.connect(
        duckdb_path, read_only=False
    )  # Ensure read_only=False for write access
    print(f"✅ DuckDB database created at: {duckdb_path}")

    # Set the temporary directory for DuckDB to the duckdb_files directory, this
    # allows DuckDB to store files on disk when performing operations
    # exceed RAM memory limits
    con.execute(f"SET temp_directory='{duckdb_dir}'")
    print(f"✅ Temporary directory for DuckDB set to: {duckdb_dir}")

    return con


CON = set_up_duckdb(CWD)

✅ DuckDB files directory is at: /mnt/ceph/vol_02_home_students/studraabf1/PhD/MRI_data_analysis/Data preprocessing + pre-analysis/Pipeline-V2/Determine power threshold_opt/duckdb_files
✅ DuckDB database created at: /mnt/ceph/vol_02_home_students/studraabf1/PhD/MRI_data_analysis/Data preprocessing + pre-analysis/Pipeline-V2/Determine power threshold_opt/duckdb_files/MRI_data.duckdb
✅ Temporary directory for DuckDB set to: /mnt/ceph/vol_02_home_students/studraabf1/PhD/MRI_data_analysis/Data preprocessing + pre-analysis/Pipeline-V2/Determine power threshold_opt/duckdb_files


In [ ]:
def get_mri_folders(data_dir: str, dir_exclude: list = []) -> list:
    """
    The data of the MRI scanners is stored in dedictaed folders for each MRI
    scanner. For further processing get a list of all MRI folder. Always
    exclude the DuckDB folder and CSV files which were created to allow the
    merging. Additionally, the user can specify other folders to exclude by
    providing a list of folder names to exclude.

    Parameters:
        data_dir (str): Path to the data directory.
        dir_exclude (list): List of directory names to exclude.

    Returns:
        list: List of MRI folder names.
    """

    # Get the list of all folders which should correspond to MRI scanners.
    # Exclude the folder that contains the DuckDB temp files and any CSV files which
    # were created to allow the merging
    return [
        folder
        for folder in os.listdir(data_dir)
        if (
            not folder.startswith("duckdb")
            and not folder.endswith(".csv")
            and folder not in dir_exclude
        )
    ]


MRI_FOLDERS = get_mri_folders(DATA_DIR)
print(MRI_FOLDERS)

['ukt_142082', 'ukt_142185', 'ukt_167008', 'ukt_183811', 'ukt_202017', 'ukt_69667', 'ukt_75609']


In [ ]:
def check_mri_folder(
    data_dir: str, con: dd.DuckDBPyConnection, mri_folders: list
) -> list:
    """
    Control if the data directory contains folders corresponding to MRI scanners by
    checking if the given Scanner.csv file contains a 'MR' pattern in the
    SiteSecondaryName column which indicates that the parent folder corresponds
    to an MRI scanner.

    Parameters:
        data_dir (str): Path to the data directory.
        con (dd.DuckDBPyConnection): A connection object to the DuckDB database.
        mri_folders (list): List of MRI folder names.

    Returns:
        list: List of valid MRI folder names.
    """
    # Initialize an empty list to store valid MRI folder names
    valid_mri_folders = []

    # Iterate over MRI folders
    for mri_folder in tqdm(mri_folders, desc="Checking MRI folders"):

        # Construct the path to the MRI folder
        mri_folder_path = os.path.join(data_dir, mri_folder)

        # Get the distinct SiteSecondaryName values from all Scanner.csv files in
        # the MRI folder
        scanner_df = con.execute(
            f"""
            SELECT DISTINCT
                SiteSecondaryName
            FROM 
                read_csv_auto('{mri_folder_path}/*/Scanner.csv', union_by_name=true)
            WHERE 
                SiteSecondaryName IS NOT NULL AND
                SiteSecondaryName LIKE 'MR%'
            """
        ).df()

        # Check if the column SiteSecondaryName is empty in the DataFrame, which
        # indicates that no MRI scanner was found
        if scanner_df.empty:
            raise ValueError(
                f"""
                Folder {mri_folder} does not contain any Scanner.csv files with 
                a 'MR' pattern in the SiteSecondaryName column. Indicating
                that this folder does not correspond to an MRI scanner. Please
                check the folder structure and ensure that only MRI scanner folders
                are present in the data directory.
                """
            )
        else:
            valid_mri_folders.append(mri_folder)

    print(
        f"✅ No invalid MRI folders found.\nValid MRI folders:\n"
        + "\n".join(f"\t📂 {folder}" for folder in valid_mri_folders)
    )

    return valid_mri_folders


VALID_MRI_FOLDERS = check_mri_folder(DATA_DIR, CON, MRI_FOLDERS)

Checking MRI folders:   0%|          | 0/7 [00:00<?, ?it/s]

✅ No invalid MRI folders found.
Valid MRI folders:
	📂 ukt_142082
	📂 ukt_142185
	📂 ukt_167008
	📂 ukt_183811
	📂 ukt_202017
	📂 ukt_69667
	📂 ukt_75609


In [ ]:
def create_mapping_scanner_mapping(data_dir: str, mri_folders: list) -> pd.DataFrame:
    """
    Create a mapping between ScannerID, Serial, and PowermeterID by iterating over
    the MRI folders. Extract the ScannerID and Serial from the Scanner.csv files,
    get the filenames of the powerdata CSV files which contain the PowermeterID
    and comile this information into a DataFrame. The mapping is saved to a
    CSV file in the data directory and also registered as a DuckDB table for
    further use in SQL queries.

    Parameters:
        data_dir (str): Path to the data directory.
        mri_folders (list): List of MRI folder names.

    Returns:
        pd.DataFrame: A DataFrame containing the mapping between ScannerID, Serial, and PowermeterID.
    """

    # Initialize an empty list to store the mappings
    mappings = []

    # Iterate over MRI folders, use tqdm to show progress
    for mri_folder in tqdm(mri_folders, desc="Processing MRI folders"):

        # Get unique ScannerID and Serial assuming one per folder
        scanner_df = con.execute(
            f"""
            SELECT DISTINCT 
                ScannerID,
                Serial
            FROM 
                read_csv_auto('{data_dir}/{mri_folder}/*/Scanner.csv', union_by_name=true)
            """
        ).df()

        # Skip if no scanner data
        if scanner_df.empty:
            warnings.warn(f"No scanner data found in {mri_folder}, skipping...")
            continue

        # Get unique ScannerIDs, which should be one per folder
        scanner_ids = scanner_df["ScannerID"].unique()
        # Get unique Serials, which should be one per folder
        serials = scanner_df["Serial"].unique()

        # If there are multiple unique ScannerIDs or Serials,
        # raise an error since we expect only 1
        if len(scanner_ids) > 1 or len(serials) > 1:
            raise ValueError(
                f"Multiple unique ScannerIDs/Serials in {mri_folder}: "
                f"{scanner_ids}, {serials}"
            )

        # Get the single ScannerID and Serial for the current MRI folder
        scanner_id = scanner_ids[0]
        serial = serials[0]

        # Get powerdataID by scanning the folder structure for
        # powerdata CSV files which should be in the format powerdata_{PowermeterID}.csv
        power_files = glob.glob(
            os.path.join(data_dir, mri_folder, "*", "powerdata_*.csv")
        )

        # Skip if no power data files found
        if not power_files:
            warnings.warn(f"No power data files found in {mri_folder}, skipping...")
            continue

        # Extract PowermeterID from filename by splitting the filename
        # assuming format powerdata_{PowermeterID}.csv
        power_ids = {
            os.path.splitext(os.path.basename(f))[0].split("_", 1)[1]
            for f in power_files
        }

        # Create mappings for each PowermeterID
        for power_id in power_ids:
            mappings.append(
                {
                    "ScannerID": scanner_id,
                    "Serial": serial,
                    "PowermeterID": power_id,
                }
            )

    # Create DataFrame and save to CSV, ensuring to drop duplicates and sort for
    # better readability
    df = (
        pd.DataFrame(mappings)
        .drop_duplicates()
        .sort_values(by=["ScannerID", "Serial", "PowermeterID"])
        .reset_index(drop=True)
    )

    # Save the mapping to a CSV file in the data directory
    output_path = os.path.join(data_dir, "powermeter_scanner_mapping.csv")
    df.to_csv(output_path, index=False, sep=";")

    # Register the DataFrame as a DuckDB table for further use in SQL queries
    con.register("powermeter_scanner_mapping", df)

    return df


POWERMETER_SCANNER_MAPPING_DF = create_mapping_scanner_mapping(DATA_DIR, VALID_MRI_FOLDERS)
display(POWERMETER_SCANNER_MAPPING_DF)

Processing MRI folders:   0%|          | 0/7 [00:00<?, ?it/s]

,ScannerID,Serial,PowermeterID
0,50,142185,LQN230413610175
1,51,142082,LQN230413610154
...,...,...,...
6,55,75609,LQN230413610162
7,56,202017,LQN201008610016


In [ ]:
def get_powerdata(
    data_dir: str,
    scanner_id: str,
    date: str,
) -> pd.DataFrame:
    """
    Get the powerdata_*.csv files from all MRI folders for a given scanner ID and date,
    join them into a single DataFrame.

    Parameters:
        data_dir (str): Path to the data directory.
        scanner_id (str): ID of the scanner.
        date (str): Date for filtering data, in the format 'YYYY-MM-DD'.

    Returns:
        pd.DataFrame: A DataFrame containing the power data.
    """

    power_df = con.execute(
        f"""
        SELECT
            powermeter_scanner_mapping.ScannerID,
            powermeter_scanner_mapping.Serial,
            power.*
        FROM
            read_csv_auto('{data_dir}/{scanner_id}/{date}/powerdata_*.csv', union_by_name=true) AS power
        INNER JOIN
            powermeter_scanner_mapping
        ON
            power.FK_PowermeterSerial = powermeter_scanner_mapping.PowermeterID
        """
    ).df()

    con.register("power", power_df)

    return power_df

In [ ]:
def enrich_powerdata(power_df: pd.DataFrame) -> pd.DataFrame:
    """
    Enrich the power data by creating new columns for total energy, apparent power,
    active power, and reactive power based on the existing columns. This involves
    summing up the energy and power values across different phases (L1, L2, L3)
    to create unified columns that represent the total values.

    Parameters:
        power_df (pd.DataFrame): The original power data DataFrame to be enriched.

    Returns:
        pd.DataFrame: The enriched power data DataFrame with new columns for total energy,
                    apparent power, active power, reactive power, and a date key
                    for daily aggregation.
    """

    # Create TotalEnergy_KWh by adding up the different phases of
    # TotalEnergyL1_KWh, TotalEnergyL2_KWh, TotalEnergyL3_KWh
    power_df["TotalEnergy_KWh"] = (
        power_df["TotalEnergyL1_KWh"].fillna(0)
        + power_df["TotalEnergyL2_KWh"].fillna(0)
        + power_df["TotalEnergyL3_KWh"].fillna(0)
    )

    # Create TotalApparentPower_KVA by adding up the different phases of
    # ApparentPowerL1_VA, ApparentPowerL2_VA, ApparentPowerL3_VA and
    # converting from VA to KVA
    power_df["TotalApparentPower_KVA"] = (
        power_df["ApparentPowerL1_VA"].fillna(0)
        + power_df["ApparentPowerL2_VA"].fillna(0)
        + power_df["ApparentPowerL3_VA"].fillna(0)
    ) / 1000

    # Create TotalActivePower_KW by adding up the different phases of
    # ActivePowerL1_W, ActivePowerL2_W, ActivePowerL3_W and converting from W to KW
    power_df["TotalActivePower_KW"] = (
        power_df["ActivePowerL1_W"].fillna(0)
        + power_df["ActivePowerL2_W"].fillna(0)
        + power_df["ActivePowerL3_W"].fillna(0)
        # Convert from W to KW
    ) / 1000

    # Create TotalReactivePower_KVAR by taking the square root of the difference between the
    # square of TotalApparentPower_KVA and the square of TotalActivePower_KW
    power_df["TotalReactivePower_KVAR"] = (
        power_df["TotalApparentPower_KVA"] ** 2 - power_df["TotalActivePower_KW"] ** 2
    ).pow(0.5)

    # Register the powerdata DataFrame as a DuckDB table
    con.register("power", power_df)

    return power_df

In [ ]:
def groupby_downsample_power_df(
    power_df: pd.DataFrame,
    downsample: int,
) -> pd.DataFrame:
    """
    Group the power and energy data by the scanner serial and time. Grouping by
    the serial has to be done since some scanners have multiple powermeters
    and thus multiple entries per time point. Take the mean of the power and
    add up the energy. Downsample the data by taking the every nth value
    of the energy data based on the downsample parameter.

    Parameters:
        power_df (pd.DataFrame): The enriched power data DataFrame to be grouped and downsampled.
        downsample (int): The downsampling factor, indicating how many energy data points to
        skip between each mean calculation.
    Returns:
        pd.DataFrame: The grouped and downsampled power and energy data DataFrame with mean power values
                    and downsampled energy values.
    """

    # Group the power data by the scanner serial and time,
    # take the mean of the power values and sum the energy values
    power_grouped = (
        power_df.groupby(["Serial", "Time"])
        .agg(
            {
                "TotalApparentPower_KVA": "mean",
                "TotalActivePower_KW": "mean",
                "TotalReactivePower_KVAR": "mean",
                "TotalEnergy_KWh": "sum",
            }
        )
        .reset_index()
    )

    # Downsample by taking every nth value of the energy data based on the downsample parameter
    downsampled = power_grouped.iloc[::downsample, :].copy()

    return downsampled

In [ ]:
def plot_powerpatterns(power_df: pd.DataFrame, scanner_id: int) -> None:
    """
    Plot the power during the sampled time interval for a given scanner serial number.

    Parameters:
        power_df (pd.DataFrame): DataFrame containing the power data.
        scanner_id (int): The serial number of the scanner

    Returns:
        None: Displays a Plotly figure with the weekly power patterns.
    """

    # Sort power_df by Time to ensure the data is in chronological order for plotting
    power_df = power_df.sort_values(by="Time")

    # Plot the Dates vs the Power consumption for the specified scanner serial number and date range
    fig = go.Figure()
    fig.add_trace(
        go.Scattergl(
            x=power_df["Time"],
            y=power_df["TotalActivePower_KW"],
            mode="lines",
            name="Total Active Power (KW)",
            marker=dict(size=3),
        )
    )

    fig.update_layout(
        title=f"Power Patterns for Scanner ID: {scanner_id}",
        xaxis_title="Time",
        yaxis_title="Total Active Power (KW)",
    )

    fig.show()

    save_fig_formats(fig, f"Power_Patterns_ScannerID_{scanner_id}")

In [ ]:
def proccess_plot_powerpatterns(scanner_ids: list, date: str) -> None:
    """
    Process the power data for a given scanner ID list and data, and plot the
    power patterns.

    Parameters:
        scanner_id (list): List of scanner IDs to process.
        date (str): The date for which to process the power data, in the format
                    'YYYY-MM-DD'.
    Returns:
        None: Displays a Plotly figure with the power patterns for the specified scanner and date.
    """

    # Iterate over the scanner IDs
    for scanner_id in tqdm(scanner_ids, desc="Processing scanners"):

        # Get and enrich the power data for the current scanner ID and date
        power_df = get_powerdata(data_dir, scanner_id, date)
        power_df = enrich_powerdata(power_df)
        print(f"Shape of the original power_df: {power_df.shape}")

        # Groupby the scanner, time and downsample the power data to simplify the plotting
        downsampled_df = groupby_downsample_power_df(power_df, downsample=100)
        print(f"Shape of the downsampled_df: {downsampled_df.shape}")

        # Plot the power patterns for the current scanner ID
        plot_powerpatterns(downsampled_df, scanner_id)


SCANNER_IDS = [
    "ukt_69667",
    "ukt_75609",
    "ukt_142082",
    "ukt_142185",
    "ukt_167008",
    "ukt_183811",
    "ukt_202017",
]

DATE = "2025-07-**"

proccess_plot_powerpatterns(SCANNER_IDS, DATE)

Processing scanners:   0%|          | 0/7 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape of the original power_df: (2604682, 31)
Shape of the downsampled_df: (26046, 6)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape of the original power_df: (5209367, 31)
Shape of the downsampled_df: (26146, 6)


...

Shape of the original power_df: (2604684, 31)
Shape of the downsampled_df: (26043, 6)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape of the original power_df: (2604683, 31)
Shape of the downsampled_df: (26044, 6)


Shape of the original power_df: (2604683, 31)
Shape of the downsampled_df: (26047, 6)


Shape of the original power_df: (2604684, 31)
Shape of the downsampled_df: (26044, 6)


In [ ]:
# Close the DuckDB connection after use
CON.close()

# Heuristics for Determining Scanner Mode 🩻

## Scanner Modes
We can use **Total Active Power (kW)** to infer the current mode of the scanner <br>
during a time interval of a month. The scanner modes are as follows:

- **Scanning Mode**: Total power above `IdleBoundary_kW` ⚡
- **Idle Mode**: Total power between `EcoPowerModeBoundary_kW` and `IdleBoundary_kW` 💤
- **Eco-Power-Mode (EPM)**: Total power below `EcoPowerModeBoundary_kW` 🌱

### Inferring Rules:
1. **Idle Levels**:
   - During the day, next to the scanning peaks, the scanner should be in **Idle Mode** 💤.
   - During the night, the **Eco-Power-Mode** 🌱 should be active.
2. **Safety Margin**:
   - Round it up, add **0.5 kW** as a safety margin to the thresholds.

### Scanners Without Eco-Power-Mode 🚫
The following scanners do not use the Eco-Power-Mode, so no threshold is set for them:
- **142082**
- **142185**
- **167008**

---

## Siemens Inferred Table 🏭

| **Serial** | **EcoPowerModeBoundary_kW 🌱** | **IdleBoundary_kW 💤** |
|------------|-------------------------------|------------------------|
| 69667      | 11.000                        | 13.000                |
| 142185     | 7.000                         | 11.000                |
| 183811     | 9.000                         | 10.500                |
| 202017     | 12.000                        | 14.000                |

---

## Personal Inferred Table 🧠

| **Serial** | **EcoPowerModeBoundary_kW 🌱** | **IdleBoundary_kW 💤** |
|------------|-------------------------------|------------------------|
| 69667      | 9.500                         | 12.500                |
| 75609      | 4.500                         | 6.500                 |
| 142082     |                               | 6.500                 |
| 142185     |                               | 6.500                 |
| 167008     |                               | 6.500                 |
| 183811     | 8.500                         | 10.500                |
| 202017     | 9.500                         | 13.500                |